# ECG Signal Discriminator

This notebook defines the **discriminator** component of the GAN (Generative Adversarial Network) used to generate synthetic ECG signals.

The discriminator's job is to look at an ECG signal and decide whether it is **real** (from the training dataset) or **fake** (generated by the generator). During training, the discriminator and generator compete: the generator tries to fool the discriminator, while the discriminator tries to catch the fakes. This adversarial dynamic is what drives both networks to improve.

---

### Two variants are defined here:

| Class | Description |
|---|---|
| `Discriminator` | Standard (unconditional) — classifies any ECG as real or fake |
| `ConditionalDiscriminator` | Label-conditioned — also receives the ECG class (Normal, AFib, Other) as input, enabling class-aware discrimination |

Both variants share the same 1D convolutional backbone that progressively compresses a 3000-sample ECG signal down to a single probability score.

## 1. Setup & Imports

We only need PyTorch and its neural network module. The default device is set to CPU so the code runs without a GPU.

In [ ]:
import torch
import torch.nn as nn

# Run all tensors on CPU by default.
# Change to 'cuda' if you have a GPU available.
torch.set_default_device('cpu')

## 2. Standard Discriminator

### Architecture

The discriminator is a **1D Convolutional Neural Network (CNN)** that treats the ECG signal as a time-series and slides learned filters across it to extract local patterns (e.g. QRS complexes, T-waves).

Each convolutional block doubles the number of feature channels while halving the sequence length via strided convolution (`stride=2`). This is the standard "encoder" pattern — compress spatial resolution, expand feature depth.

```
Input:  (batch, 1, 3000)   ← raw ECG signal, single channel
  ↓ Conv Block 1            kernel=25, stride=2
(batch, 64, 1500)
  ↓ Conv Block 2            + BatchNorm
(batch, 128, 750)
  ↓ Conv Block 3            + BatchNorm
(batch, 256, 375)
  ↓ Conv Block 4            + BatchNorm
(batch, 512, 188)
  ↓ Conv Block 5            + BatchNorm
(batch, 512, 94)
  ↓ Flatten → Linear(1) → Sigmoid
Output: (batch, 1)         ← probability of being real [0, 1]
```

### Design choices

- **LeakyReLU(0.2)**: Preferred over ReLU in discriminators — it allows small gradients for negative activations, which stabilises GAN training.
- **BatchNorm**: Normalises activations within each mini-batch, speeding up training and reducing sensitivity to weight initialisation. Omitted in the first block (common practice).
- **Dropout**: Regularisation to prevent the discriminator from memorising training samples.
- **Sigmoid output**: Squashes the final score to `[0, 1]`, interpreted as the probability that the input is a real ECG.

In [ ]:
class Discriminator(nn.Module):
    """
    Unconditional 1D-CNN discriminator for ECG signals.

    Takes a single-channel ECG signal of length `seq_length` and outputs
    a scalar probability that the signal is real (vs. generator-produced).

    Args:
        seq_length (int): Number of samples in the ECG signal. Default 3000.
        dropout (float): Dropout probability applied after each conv block. Default 0.3.
    """

    def __init__(self, seq_length=3000, dropout=0.3):
        super(Discriminator, self).__init__()

        self.seq_length = seq_length

        # --- Convolutional Blocks ---
        # Each block: Conv1d → (BatchNorm) → LeakyReLU → Dropout
        # stride=2 halves the sequence length at each stage.
        # kernel_size=25 with padding=12 keeps the math clean (no off-by-one).

        # Block 1: 1 channel  → 64 features  | length 3000 → 1500
        # (No BatchNorm on the first layer — standard discriminator practice)
        self.conv1 = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=64, kernel_size=25, stride=2, padding=12),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout)
        )

        # Block 2: 64 → 128 features | length 1500 → 750
        self.conv2 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=25, stride=2, padding=12),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout)
        )

        # Block 3: 128 → 256 features | length 750 → 375
        self.conv3 = nn.Sequential(
            nn.Conv1d(128, 256, kernel_size=25, stride=2, padding=12),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout)
        )

        # Block 4: 256 → 512 features | length 375 → 188
        self.conv4 = nn.Sequential(
            nn.Conv1d(256, 512, kernel_size=25, stride=2, padding=12),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout)
        )

        # Block 5: 512 → 512 features | length 188 → 94
        self.conv5 = nn.Sequential(
            nn.Conv1d(512, 512, kernel_size=25, stride=2, padding=12),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout)
        )

        # After 5 strided convolutions: 3000 → 1500 → 750 → 375 → 188 → 94
        # Flattened feature vector size = 94 timesteps × 512 channels
        self.flatten_size = 94 * 512

        # --- Classification Head ---
        # Flatten the feature map and project to a single probability score.
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flatten_size, 1),
            nn.Sigmoid()   # Output: probability in [0, 1]
        )

    def forward(self, x):
        """
        Forward pass.

        Args:
            x: ECG tensor. Accepted shapes:
               - (batch, seq_length, 1)  — "sequence-last" format from the dataloader
               - (batch, 1, seq_length)  — "channel-first" format expected by Conv1d

        Returns:
            Tensor of shape (batch, 1) with real/fake probabilities.
        """
        # Conv1d expects (batch, channels, length), so re-order if necessary.
        if x.shape[-1] == 1:
            x = x.permute(0, 2, 1)  # (batch, seq_length, 1) → (batch, 1, seq_length)

        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.conv5(x)
        x = self.fc(x)   # Flatten + linear + sigmoid

        return x

## 3. Conditional Discriminator

### Why condition on the class label?

In a **Conditional GAN (cGAN)**, both the generator and discriminator receive extra information: the class label of the ECG (e.g. `0 = Normal`, `1 = AFib`, `2 = Other`). This has two benefits:

1. The generator can be directed to produce a *specific* type of ECG (targeted generation).
2. The discriminator can apply class-specific standards — e.g. it knows that AFib signals should be irregularly irregular, so it can more easily spot a fake Normal signal being passed off as AFib.

### How is the label injected?

The label is turned into a continuous vector via an **embedding layer**, then stretched to match the ECG length and **concatenated as a second channel** alongside the raw signal. This gives the convolutional layers label information at every timestep.

```
Label (integer)  →  Embedding (50-d)  →  Linear projection (3000-d)  →  reshape to (1, 3000)
                                                                              ↓
ECG signal (1, 3000) ─────────────────────────────────── cat ──────→ (2, 3000)
                                                                              ↓
                                                            Same 5-block CNN as above
                                                                              ↓
                                                                      Output: (batch, 1)
```

In [ ]:
class ConditionalDiscriminator(nn.Module):
    """
    Class-conditional 1D-CNN discriminator for ECG signals.

    Extends the standard Discriminator by accepting a class label alongside
    the ECG signal. The label is embedded and concatenated as a second input
    channel before the convolutional stack.

    Args:
        seq_length   (int): Number of samples in the ECG signal. Default 3000.
        num_classes  (int): Number of ECG classes (e.g. 3 for Normal/AFib/Other).
        embedding_dim(int): Size of the label embedding vector. Default 50.
        dropout    (float): Dropout probability per conv block. Default 0.3.
    """

    def __init__(self, seq_length=3000, num_classes=3, embedding_dim=50, dropout=0.3):
        super(ConditionalDiscriminator, self).__init__()

        self.seq_length = seq_length
        self.num_classes = num_classes
        self.embedding_dim = embedding_dim

        # --- Label Conditioning Layers ---

        # Turn integer class label into a dense embedding vector.
        # E.g. label=1 → learnable 50-dimensional vector.
        self.label_embedding = nn.Embedding(num_classes, embedding_dim)

        # Stretch the 50-d embedding to the full ECG length (3000)
        # so it can be used as a second channel alongside the signal.
        self.label_proj = nn.Linear(embedding_dim, seq_length)

        # --- Convolutional Blocks ---
        # Identical to Discriminator, except Block 1 takes 2 input channels
        # (channel 0 = ECG signal, channel 1 = projected label).

        # Block 1: 2 channels → 64 features | length 3000 → 1500
        self.conv1 = nn.Sequential(
            nn.Conv1d(in_channels=2, out_channels=64, kernel_size=25, stride=2, padding=12),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout)
        )

        # Block 2: 64 → 128 features | length 1500 → 750
        self.conv2 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=25, stride=2, padding=12),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout)
        )

        # Block 3: 128 → 256 features | length 750 → 375
        self.conv3 = nn.Sequential(
            nn.Conv1d(128, 256, kernel_size=25, stride=2, padding=12),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout)
        )

        # Block 4: 256 → 512 features | length 375 → 188
        self.conv4 = nn.Sequential(
            nn.Conv1d(256, 512, kernel_size=25, stride=2, padding=12),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout)
        )

        # Block 5: 512 → 512 features | length 188 → 94
        self.conv5 = nn.Sequential(
            nn.Conv1d(512, 512, kernel_size=25, stride=2, padding=12),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout)
        )

        self.flatten_size = 94 * 512

        # --- Classification Head ---
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flatten_size, 1),
            nn.Sigmoid()
        )

    def forward(self, x, labels):
        """
        Forward pass with class conditioning.

        Args:
            x      : ECG tensor, shape (batch, seq_length, 1) or (batch, 1, seq_length).
            labels : Integer class labels, shape (batch,).

        Returns:
            Tensor of shape (batch, 1) with real/fake probabilities.
        """
        # Ensure channel-first layout for Conv1d.
        if x.shape[-1] == 1:
            x = x.permute(0, 2, 1)  # (batch, seq_length, 1) → (batch, 1, seq_length)

        # --- Build the label channel ---
        # 1. Embed: integer label → 50-d continuous vector
        label_embed = self.label_embedding(labels)       # (batch, embedding_dim)
        # 2. Project: 50-d → 3000-d (same length as the ECG signal)
        label_proj = self.label_proj(label_embed)        # (batch, seq_length)
        # 3. Unsqueeze: add a channel dimension so it matches Conv1d's input format
        label_proj = label_proj.unsqueeze(1)             # (batch, 1, seq_length)

        # Concatenate along the channel dimension:
        # [ECG channel | label channel] → 2-channel input for the CNN
        x = torch.cat([x, label_proj], dim=1)            # (batch, 2, seq_length)

        # Pass through the shared convolutional backbone
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.conv5(x)
        x = self.fc(x)

        return x

## 4. Sanity Tests

Before plugging the discriminators into the training loop, we run quick shape checks to confirm:

- Input and output tensors have the expected dimensions.
- Output values fall within `[0, 1]` (Sigmoid is working).
- Parameter counts are reported (useful for comparing model complexity).

We use a random batch of 100 signals (`torch.randn`) — the values are meaningless, but the shapes are what matter here.

In [ ]:
def test_discriminator():
    """Run shape/output sanity checks for both discriminator variants."""

    batch_size = 100
    seq_length = 3000

    # Dummy batch: random noise shaped like a real dataloader batch
    dummy_input = torch.randn(batch_size, seq_length, 1)  # (batch, length, channels)

    # ------------------------------------------------------------------
    # Test 1: Standard (unconditional) Discriminator
    # ------------------------------------------------------------------
    print("Testing Discriminator...")
    disc = Discriminator(seq_length=seq_length, dropout=0.3)

    output = disc(dummy_input)

    print(f"  Input shape  : {dummy_input.shape}")     # expect (100, 3000, 1)
    print(f"  Output shape : {output.shape}")           # expect (100, 1)
    print(f"  Output range : [{output.min().item():.4f}, {output.max().item():.4f}]")  # expect [0,1]

    total_params     = sum(p.numel() for p in disc.parameters())
    trainable_params = sum(p.numel() for p in disc.parameters() if p.requires_grad)
    print(f"  Total params    : {total_params:,}")
    print(f"  Trainable params: {trainable_params:,}")

    # ------------------------------------------------------------------
    # Test 2: Conditional Discriminator
    # ------------------------------------------------------------------
    print("\n" + "=" * 60)
    print("Testing Conditional Discriminator...")
    cond_disc = ConditionalDiscriminator(seq_length=seq_length, num_classes=3, dropout=0.3)

    # Random integer labels in {0, 1, 2} — one per sample in the batch
    dummy_labels = torch.randint(0, 3, (batch_size,))

    output = cond_disc(dummy_input, dummy_labels)

    print(f"  Input shape  : {dummy_input.shape}")     # expect (100, 3000, 1)
    print(f"  Labels shape : {dummy_labels.shape}")    # expect (100,)
    print(f"  Output shape : {output.shape}")           # expect (100, 1)
    print(f"  Output range : [{output.min().item():.4f}, {output.max().item():.4f}]")

    total_params     = sum(p.numel() for p in cond_disc.parameters())
    trainable_params = sum(p.numel() for p in cond_disc.parameters() if p.requires_grad)
    print(f"  Total params    : {total_params:,}")
    print(f"  Trainable params: {trainable_params:,}")

    print("\n✓ All tests passed!")


test_discriminator()